# Pub/Sub Hands-On Lab

A plain walkthrough of Google Cloud Pub/Sub: topics, subscriptions (pull and push), acking,
dead-letters, snapshots, and a few delivery options.



## 1. Setup

In [ ]:

from google.colab import auth
auth.authenticate_user()
print("authenticated")


In [ ]:
!pip install --quiet google-cloud-pubsub

In [ ]:
import time
import json
from google.cloud import pubsub_v1
from google.pubsub_v1.types import Schema, Encoding
from google.api_core import exceptions as api_exceptions
from google.protobuf import duration_pb2
from google.protobuf.timestamp_pb2 import Timestamp

In [ ]:
PROJECT_ID = "project-2df9d2ad-b6f5-4ed5-997"
REGION = "us-central1"

!gcloud config set project {PROJECT_ID}
!gcloud services enable pubsub.googleapis.com run.googleapis.com --project={PROJECT_ID}

suffix = "yourname1"

TOPIC_ID = f"training-topic-{suffix}"
SCHEMA_TOPIC_ID = f"training-schema-topic-{suffix}"
SCHEMA_ID = f"training-schema-{suffix}"
SUB_A_ID = f"training-sub-a-{suffix}"
SUB_B_ID = f"training-sub-b-{suffix}"
PUSH_SUB_ID = f"training-sub-push-{suffix}"
DLQ_TOPIC_ID = f"training-dlq-topic-{suffix}"
DLQ_SUB_ID = f"training-sub-dlq-{suffix}"
FILTERED_SUB_ID = f"training-sub-filtered-{suffix}"
RETRY_SUB_ID = f"training-sub-retry-{suffix}"
EXACTLY_ONCE_SUB_ID = f"training-sub-exactly-once-{suffix}"
SNAPSHOT_ID = f"training-snapshot-{suffix}"

publisher = pubsub_v1.PublisherClient()
subscriber = pubsub_v1.SubscriberClient()
print("ready, main topic will be", TOPIC_ID)

## 2. Topic

In [ ]:
topic_path = publisher.topic_path(PROJECT_ID, TOPIC_ID)
publisher.create_topic(request={"name": topic_path})

for t in publisher.list_topics(request={"project": f"projects/{PROJECT_ID}"}):
    print(" -", t.name)

> 🖥️ Pub/Sub > Topics in the console — your new topic should be listed there.

## 3. Schema registry

A schema rejects malformed messages at publish time, instead of you finding out three
steps downstream.

In [ ]:
schema_client = pubsub_v1.SchemaServiceClient()
schema_path = schema_client.schema_path(PROJECT_ID, SCHEMA_ID)

avro_definition = json.dumps({
    "type": "record", "name": "TrainingEvent",
    "fields": [
        {"name": "event_id", "type": "string"},
        {"name": "event_type", "type": "string"},
        {"name": "amount", "type": "double"},
    ],
})

schema_client.create_schema(request={
    "parent": f"projects/{PROJECT_ID}",
    "schema": {"name": schema_path, "type_": Schema.Type.AVRO, "definition": avro_definition},
    "schema_id": SCHEMA_ID,
})

schema_topic_path = publisher.topic_path(PROJECT_ID, SCHEMA_TOPIC_ID)
publisher.create_topic(request={
    "name": schema_topic_path,
    "schema_settings": {"schema": schema_path, "encoding": Encoding.JSON},
})
print("schema + enforced topic created")

In [ ]:
valid = json.dumps({"event_id": "evt-001", "event_type": "purchase", "amount": 42.5}).encode()
publisher.publish(schema_topic_path, data=valid).result()
print("valid message went through")

invalid = json.dumps({"event_id": "evt-002", "amount": "not-a-number"}).encode()
try:
    publisher.publish(schema_topic_path, data=invalid).result()
    print("that shouldn't have worked -- check the schema")
except api_exceptions.GoogleAPICallError as e:
    print("rejected as expected:", str(e)[:150])

## 4. Subscriptions: pull vs push

Every subscription is its own independent copy of the topic's messages -- that's fan-out.
Two pull subscriptions first, both on the same topic:

In [ ]:
sub_a_path = subscriber.subscription_path(PROJECT_ID, SUB_A_ID)
sub_b_path = subscriber.subscription_path(PROJECT_ID, SUB_B_ID)

subscriber.create_subscription(request={"name": sub_a_path, "topic": topic_path})
subscriber.create_subscription(request={"name": sub_b_path, "topic": topic_path})
print("A and B created, both attached to the same topic")

### Push, for real

Push means Pub/Sub calls an HTTPS endpoint for you, instead of you calling `pull()`.


In [ ]:
import os
os.makedirs("push_service", exist_ok=True)

with open("push_service/main.py", "w") as f:
    f.write('''
import os
import base64
from flask import Flask, request

app = Flask(__name__)

# just an in-memory list -- fine for a training demo, resets if the
# container restarts or Cloud Run scales to zero after idling
received_messages = []

@app.route("/", methods=["GET"])
def home():
    if not received_messages:
        return "Push endpoint is up. Nothing received yet -- publish a message and refresh this page."
    rows = "".join(
        f"<li><b>{m[\'data\']}</b> -- attributes: {m[\'attributes\']}</li>"
        for m in reversed(received_messages)
    )
    return f"<h3>Received {len(received_messages)} message(s):</h3><ul>{rows}</ul>"

@app.route("/", methods=["POST"])
def receive_push():
    envelope = request.get_json()
    if not envelope or "message" not in envelope:
        return "Bad Request: no Pub/Sub message", 400
    message = envelope["message"]
    data = base64.b64decode(message.get("data", "")).decode("utf-8") if message.get("data") else ""
    attributes = message.get("attributes", {})
    print(f"Received push message: data={data!r} attributes={attributes}")
    received_messages.append({"data": data, "attributes": attributes})
    return "", 204

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.environ.get("PORT", 8080)))
''')

with open("push_service/requirements.txt", "w") as f:
    f.write("flask==3.*\ngunicorn==22.*\n")

print("push_service/ written")

In [ ]:
SERVICE_NAME = f"training-push-handler-{suffix}"

!gcloud run deploy {SERVICE_NAME} --source=push_service --region={REGION} \
    --project={PROJECT_ID} --allow-unauthenticated --quiet

push_url = !gcloud run services describe {SERVICE_NAME} --region={REGION} \
    --project={PROJECT_ID} --format="value(status.url)"
PUSH_ENDPOINT_URL = push_url[0].strip()
print("deployed at", PUSH_ENDPOINT_URL)
print("open that URL in a browser -- you should see 'Push endpoint is up.'")

> 🖥️ Try it: open `PUSH_ENDPOINT_URL` (printed above) in your browser. No login prompt,
> no auth error -- just the plain text response, because the service is public.

In [ ]:
push_sub_path = subscriber.subscription_path(PROJECT_ID, PUSH_SUB_ID)
subscriber.create_subscription(request={
    "name": push_sub_path,
    "topic": topic_path,
    "push_config": {"push_endpoint": PUSH_ENDPOINT_URL},
})
print("push subscription wired up to", PUSH_ENDPOINT_URL)

In [ ]:
publisher.publish(topic_path, data=b"pushed via Cloud Run!-message2").result()
print("published -- refresh PUSH_ENDPOINT_URL in your browser to see it show up")
time.sleep(15)

!gcloud run services logs read {SERVICE_NAME} --region={REGION} --project={PROJECT_ID} --limit=10

Look for `Received push message: data='pushed via Cloud Run!'` in the log output --
that's Pub/Sub itself calling your endpoint, not anything simulated locally.

## 5. Pull, ack, nack

In [ ]:
response = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 10})
print(f"pulled {len(response.received_messages)} on A")
for rm in response.received_messages:
    print(" -", rm.message.data[:60])

In [ ]:
ack_ids = [rm.ack_id for rm in response.received_messages]
if ack_ids:
    subscriber.acknowledge(request={"subscription": sub_a_path, "ack_ids": ack_ids})
    print(f"acked {len(ack_ids)}")
else:
    print("nothing pulled -- publish more first")

In [ ]:
# nack = reset the ack deadline to 0, forcing immediate redelivery
publisher.publish(topic_path, data=b"message to be nacked").result()
time.sleep(2)

resp = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 1})
if resp.received_messages:
    nack_id = resp.received_messages[0].ack_id
    subscriber.modify_ack_deadline(request={"subscription": sub_a_path, "ack_ids": [nack_id], "ack_deadline_seconds": 0})
    print("nacked -- should come right back")
else:
    print("nothing pulled -- may already be consumed")

In [ ]:
# B never acked anything -- it should still have its own untouched copies
response_b = subscriber.pull(request={"subscription": sub_b_path, "max_messages": 20})
print(f"B still has {len(response_b.received_messages)} pending, independent of whatever happened to A")

ack_ids_b = [rm.ack_id for rm in response_b.received_messages]
if ack_ids_b:
    subscriber.acknowledge(request={"subscription": sub_b_path, "ack_ids": ack_ids_b})

## 6. Dead-letter topics

After N failed delivery attempts, forward the message to a separate topic instead of
retrying forever.

In [ ]:
dlq_topic_path = publisher.topic_path(PROJECT_ID, DLQ_TOPIC_ID)
publisher.create_topic(request={"name": dlq_topic_path})

project_number = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
project_number = project_number[0].strip()
pubsub_service_agent = f"serviceAccount:service-{project_number}@gcp-sa-pubsub.iam.gserviceaccount.com"

!gcloud pubsub topics add-iam-policy-binding {DLQ_TOPIC_ID} --project={PROJECT_ID} \
    --member="{pubsub_service_agent}" --role="roles/pubsub.publisher"

print("dead-letter topic created, Pub/Sub granted publish rights on it")

In [ ]:
dlq_sub_path = subscriber.subscription_path(PROJECT_ID, DLQ_SUB_ID)
subscriber.create_subscription(request={
    "name": dlq_sub_path,
    "topic": topic_path,
    "dead_letter_policy": {"dead_letter_topic": dlq_topic_path, "max_delivery_attempts": 5},
})

!gcloud pubsub subscriptions add-iam-policy-binding {DLQ_SUB_ID} --project={PROJECT_ID} \
    --member="{pubsub_service_agent}" --role="roles/pubsub.subscriber"

print("dlq-backed subscription ready")

In [ ]:
publisher.publish(topic_path, data=b"this message will be dead-lettered").result()

for attempt in range(6):
    time.sleep(5)
    resp = subscriber.pull(request={"subscription": dlq_sub_path, "max_messages": 1})
    if not resp.received_messages:
        continue
    ack_id = resp.received_messages[0].ack_id
    subscriber.modify_ack_deadline(request={"subscription": dlq_sub_path, "ack_ids": [ack_id], "ack_deadline_seconds": 0})
    print(f"attempt {attempt}: nacked again")

dlq_check_path = subscriber.subscription_path(PROJECT_ID, f"{DLQ_SUB_ID}-check")
subscriber.create_subscription(request={"name": dlq_check_path, "topic": dlq_topic_path})
dlq_resp = subscriber.pull(request={"subscription": dlq_check_path, "max_messages": 5})
print(f"\n{len(dlq_resp.received_messages)} message(s) landed on the dead-letter topic")
for rm in dlq_resp.received_messages:
    print(" -", rm.message.data)
    subscriber.acknowledge(request={"subscription": dlq_check_path, "ack_ids": [rm.ack_id]})

## 7. Snapshots & replay

A snapshot freezes a subscription's ack state, so you can seek back to it later and
replay everything that happened since.

In [ ]:
snapshot_path = subscriber.snapshot_path(PROJECT_ID, SNAPSHOT_ID)
subscriber.create_snapshot(request={"name": snapshot_path, "subscription": sub_a_path})
print("snapshot taken")

In [ ]:
for i in range(3):
    publisher.publish(topic_path, data=f"post-snapshot {i}".encode()).result()
time.sleep(2)

# drain everything first so the replay below is unambiguous
drain = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 50})
if drain.received_messages:
    subscriber.acknowledge(request={"subscription": sub_a_path, "ack_ids": [rm.ack_id for rm in drain.received_messages]})

subscriber.seek(request={"subscription": sub_a_path, "snapshot": snapshot_path})
replayed = subscriber.pull(request={"subscription": sub_a_path, "max_messages": 50})
print(f"replayed {len(replayed.received_messages)} message(s) after seeking back")
if replayed.received_messages:
    subscriber.acknowledge(request={"subscription": sub_a_path, "ack_ids": [rm.ack_id for rm in replayed.received_messages]})

In [ ]:
# seek also takes a plain timestamp -- replay the last 10 minutes without a snapshot
ten_min_ago = Timestamp()
ten_min_ago.FromSeconds(int(time.time()) - 600)
subscriber.seek(request={"subscription": sub_a_path, "time": ten_min_ago})
print("sought back 10 minutes")

## 8. Flow control & retry policy

In [ ]:
# caps how much a slow subscriber can have in flight at once
flow_control = pubsub_v1.types.FlowControl(max_messages=5, max_bytes=1024 * 1024)
received = []

def slow_callback(message):
    received.append(message.data)
    time.sleep(1)  # pretend this is slow
    message.ack()

for i in range(8):
    publisher.publish(topic_path, data=f"flow-control {i}".encode()).result()

future = subscriber.subscribe(sub_b_path, callback=slow_callback, flow_control=flow_control)
time.sleep(8)
future.cancel()
try:
    future.result(timeout=5)
except Exception:
    pass
print(f"processed {len(received)} under flow control (max 5 in flight)")

In [ ]:
# server-side backoff for redelivery -- no client code needed to benefit from it
retry_sub_path = subscriber.subscription_path(PROJECT_ID, RETRY_SUB_ID)
subscriber.create_subscription(request={
    "name": retry_sub_path,
    "topic": topic_path,
    "retry_policy": {
        "minimum_backoff": duration_pb2.Duration(seconds=10),
        "maximum_backoff": duration_pb2.Duration(seconds=600),
    },
})
print("retry policy set: 10s-600s backoff")

## 9. Filtering

A subscription can filter on attributes server-side -- non-matching messages get
auto-acked before your code ever sees them.

In [ ]:
filtered_sub_path = subscriber.subscription_path(PROJECT_ID, FILTERED_SUB_ID)
subscriber.create_subscription(request={
    "name": filtered_sub_path, "topic": topic_path, "filter": 'attributes.priority = "high"'
})

publisher.publish(topic_path, data=b"low priority event", priority="low").result()
publisher.publish(topic_path, data=b"high priority event", priority="high").result()
time.sleep(2)

resp = subscriber.pull(request={"subscription": filtered_sub_path, "max_messages": 10})
print(f"got {len(resp.received_messages)} (expect just the high-priority one)")
for rm in resp.received_messages:
    print(" -", rm.message.data, rm.message.attributes.get("priority"))
if resp.received_messages:
    subscriber.acknowledge(request={"subscription": filtered_sub_path, "ack_ids": [rm.ack_id for rm in resp.received_messages]})

## 10. Exactly-once delivery

By default Pub/Sub is at-least-once -- rare redelivery is possible even after a
successful ack. Turning this on makes a successful ack an actual guarantee.

In [ ]:
eo_sub_path = subscriber.subscription_path(PROJECT_ID, EXACTLY_ONCE_SUB_ID)
subscriber.create_subscription(request={
    "name": eo_sub_path, "topic": topic_path, "enable_exactly_once_delivery": True
})

publisher.publish(topic_path, data=b"exactly-once demo").result()
time.sleep(2)

resp = subscriber.pull(request={"subscription": eo_sub_path, "max_messages": 1})
if resp.received_messages:
    subscriber.acknowledge(request={"subscription": eo_sub_path, "ack_ids": [resp.received_messages[0].ack_id]})
    print("acked -- with exactly-once on, this ack is a real guarantee")
else:
    print("nothing pulled yet, try again")

## Cleanup

In [ ]:
all_subs = [
    SUB_A_ID, SUB_B_ID, PUSH_SUB_ID, DLQ_SUB_ID, f"{DLQ_SUB_ID}-check",
    RETRY_SUB_ID, FILTERED_SUB_ID, EXACTLY_ONCE_SUB_ID,
]
for sub_id in all_subs:
    try:
        subscriber.delete_subscription(request={"subscription": subscriber.subscription_path(PROJECT_ID, sub_id)})
    except api_exceptions.NotFound:
        pass

subscriber.delete_snapshot(request={"snapshot": snapshot_path})

for t in [topic_path, schema_topic_path, dlq_topic_path]:
    publisher.delete_topic(request={"topic": t})

schema_client.delete_schema(request={"name": schema_path})
print("subscriptions, snapshot, topics, schema -- all gone")

In [ ]:
!gcloud run services delete {SERVICE_NAME} --region={REGION} --project={PROJECT_ID} --quiet
print("Cloud Run service deleted too")